# Notebook 7 – Binning and Discretization

This notebook covers how to convert continuous numeric data into categories (bins), why we do it, and the different methods used to do it.

## 1. What is Binning?

**Definition:** Binning means grouping continuous numbers into a smaller number of ranges, called bins.

**Example:** Age 24 becomes "18-30". Age 52 becomes "46-60".

**Why it is used?** Makes large numeric ranges easier to read, reduces noise, and helps some models work better with categories instead of raw numbers.

In [2]:
import pandas as pd
df = pd.read_csv('data.csv', encoding='latin1')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## 2. Why Binning?

**Definition:** Binning is used when raw continuous values are too detailed, spread out, or noisy to analyze directly.

**Example:** UnitPrice values like 1.25, 1.30, 1.28 are all basically "low price", so grouping them removes tiny meaningless differences.

**Why it is used?** Helps spot trends, handles outliers better, and turns messy numbers into clean, readable groups for reports and models.

In [3]:
df['UnitPrice'].describe()

count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

**Code Explanation:** describe gives min, max, mean, and spread of UnitPrice so we know what range we are binning.

## 3. Equal-Width Binning

**Definition:** Splits the full range of values into bins of equal size.

**Example:** Prices from 0 to 100 split into 5 bins of width 20 each: 0-20, 20-40, 40-60, 60-80, 80-100.

**Why it is used?** Simple and quick way to bin data when you just want equal-sized ranges, regardless of how many values fall in each.

In [4]:
df['Price_EqualWidth'] = pd.cut(df['UnitPrice'], bins=5)
df['Price_EqualWidth'].value_counts()

Price_EqualWidth
(-1055.648, 8950.764]      541897
(8950.764, 18957.176]           9
(-11112.092, -1055.648]         2
(28963.588, 38970.0]            1
(18957.176, 28963.588]          0
Name: count, dtype: int64

**Code Explanation:** pd.cut splits UnitPrice into 5 equal-width ranges. value_counts shows how many rows fall in each bin.

## 4. Equal-Frequency Binning

**Definition:** Splits data so each bin has roughly the same number of records, not the same width.

**Example:** If you have 100 rows and want 4 bins, each bin gets about 25 rows, even if the price ranges are uneven.

**Why it is used?** Useful when data is unevenly spread, so no single bin ends up overloaded or empty.

In [5]:
df['Price_EqualFreq'] = pd.qcut(df['UnitPrice'], q=4)
df['Price_EqualFreq'].value_counts()

Price_EqualFreq
(-11062.061, 1.25]    167385
(2.08, 4.13]          140503
(4.13, 38970.0]       128915
(1.25, 2.08]          105106
Name: count, dtype: int64

**Code Explanation:** pd.qcut splits UnitPrice into 4 bins so each bin holds the same number of rows.

## 5. Quantile Binning

**Definition:** A specific type of equal-frequency binning based on percentiles like 25 percent, 50 percent, 75 percent.

**Example:** Splitting Quantity into Low, Medium, High based on the 25th, 50th, and 75th percentile values.

**Why it is used?** Good for ranking or scoring data relative to the rest of the dataset, like putting customers into spending tiers.

In [6]:
df['Quantity_Quantile'] = pd.qcut(df['Quantity'], q=[0, 0.25, 0.5, 0.75, 1], labels=['Low', 'Medium', 'High', 'Very High'])
df['Quantity_Quantile'].value_counts()

Quantity_Quantile
Low          158851
Very High    132631
High         131477
Medium       118950
Name: count, dtype: int64

**Code Explanation:** qcut with custom quantile edges creates 4 labeled groups based on where each value ranks in the data.

## 6. Domain-Based Binning

**Definition:** Bins created using real-world knowledge or rules, not just statistics.

**Example:** Age groups like Child, Teen, Adult, Senior are based on common sense, not on splitting the data evenly.

**Why it is used?** Makes bins meaningful to humans and matches how the business or industry already thinks about the data.

In [7]:
ages = pd.Series([5, 17, 25, 40, 65, 72])
bins = [0, 12, 19, 60, 100]
labels = ['Child', 'Teen', 'Adult', 'Senior']
age_groups = pd.cut(ages, bins=bins, labels=labels)
age_groups

0     Child
1      Teen
2     Adult
3     Adult
4    Senior
5    Senior
dtype: category
Categories (4, str): ['Child' < 'Teen' < 'Adult' < 'Senior']

**Code Explanation:** Custom bin edges and labels are defined using known age-group cutoffs, then pd.cut applies them to the ages.

## 7. Continuous to Categorical

**Definition:** The overall process of turning number columns into label or category columns using binning.

**Example:** UnitPrice column (numbers) becomes a Price_Level column (Low, Medium, High).

**Why it is used?** Categories are easier to filter, group, and visualize than raw continuous numbers.

In [8]:
df['Price_Level'] = pd.cut(df['UnitPrice'], bins=3, labels=['Low', 'Medium', 'High'])
df[['UnitPrice', 'Price_Level']].head()

,UnitPrice,Price_Level
0,2.55,Low
1,3.39,Low
2,2.75,Low
3,3.39,Low
4,3.39,Low


**Code Explanation:** UnitPrice is split into 3 equal-width bins and labeled Low, Medium, High, creating a new categorical column.

## 8. Age Groups

**Definition:** Binning applied specifically to age data to create readable age brackets.

**Example:** Ages 18-30 as Young, 31-50 as Middle-Aged, 51+ as Senior.

**Why it is used?** Common in customer analysis and healthcare to group people by life stage instead of exact age.

In [9]:
ages = pd.Series([15, 22, 34, 45, 58, 70])
bins = [0, 17, 30, 50, 100]
labels = ['Minor', 'Young', 'Middle-Aged', 'Senior']
pd.cut(ages, bins=bins, labels=labels)

0          Minor
1          Young
2    Middle-Aged
3    Middle-Aged
4         Senior
5         Senior
dtype: category
Categories (4, str): ['Minor' < 'Young' < 'Middle-Aged' < 'Senior']

**Code Explanation:** Age values are grouped into 4 labeled brackets using fixed real-world cutoffs.

## 9. Income Groups

**Definition:** Binning applied to income or price data to separate low, medium, and high earners or spenders.

**Example:** Grouping customers by total spend into Low Spender, Medium Spender, High Spender.

**Why it is used?** Helps businesses target different customer segments with different offers.

In [10]:
spend = df.groupby('CustomerID')['UnitPrice'].sum()
spend_groups = pd.qcut(spend, q=3, labels=['Low Spender', 'Medium Spender', 'High Spender'])
spend_groups.head()

CustomerID
12346.0       Low Spender
12347.0      High Spender
12348.0    Medium Spender
12349.0      High Spender
12350.0       Low Spender
Name: UnitPrice, dtype: category
Categories (3, str): ['Low Spender' < 'Medium Spender' < 'High Spender']

**Code Explanation:** Total spend per customer is calculated, then split into 3 equal-sized spender groups using qcut.

## 10. Risk Categories

**Definition:** Binning used to convert a risk score or similar numeric measure into risk levels.

**Example:** A risk score from 0 to 100 becomes Low Risk, Medium Risk, High Risk.

**Why it is used?** Common in finance and insurance to quickly flag which customers or transactions need more attention.

In [11]:
risk_scores = pd.Series([10, 35, 55, 70, 90])
bins = [0, 30, 60, 100]
labels = ['Low Risk', 'Medium Risk', 'High Risk']
pd.cut(risk_scores, bins=bins, labels=labels)

0       Low Risk
1    Medium Risk
2    Medium Risk
3      High Risk
4      High Risk
dtype: category
Categories (3, str): ['Low Risk' < 'Medium Risk' < 'High Risk']

**Code Explanation:** Risk scores are split into 3 labeled risk levels using fixed thresholds.

## Advantages and Limitations of Binning

**Advantages**
- Simplifies complex numeric data into readable groups
- Reduces the effect of outliers and small noise
- Makes visualizations and reports clearer
- Some models handle categories better than raw numbers

**Limitations**
- Can lose detail and precision from the original data
- Bin edges chosen poorly can hide real patterns
- Different binning methods can give different results on the same data
- Adds an extra step that needs to be redone if new data comes in